In [1]:
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
import transformers
import datasets
import torch
import pandas as pd
from tqdm.auto import tqdm
import pickle
from transformer_lens import HookedTransformer, utils
import einops
import pickle
import os
from datetime import datetime

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

left_tokenizer = AutoTokenizer.from_pretrained("microsoft/Llama2-7b-WhoIsHarryPotter")
left_tokenizer.pad_token = left_tokenizer.eos_token
left_tokenizer.padding_side = "left"

right_tokenizer = AutoTokenizer.from_pretrained("microsoft/Llama2-7b-WhoIsHarryPotter")
right_tokenizer.pad_token = right_tokenizer.eos_token

# load models
# hp_model = AutoModelForCausalLM.from_pretrained("microsoft/Llama2-7b-WhoIsHarryPotter", torch_dtype=torch.bfloat16).cuda()
# regular_model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b-chat-hf", torch_dtype=torch.bfloat16).cuda()
# can also load leace, other models

# lat_model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b-chat-hf", torch_dtype=torch.bfloat16)
# lat_model = PeftModel.from_pretrained(lat_model, "models/hp-lat-llama-genericized_diff_hp_indices-2024-04-03-09-33-10")

lat_models = {}
# lat_model_names = {"LAT_Generic_Diff_HP_Indices": "models/hp-lat-llama-genericized_diff_hp_indices-2024-04-03-09-33-10", "LAT_Generic_Diff_All_Indices": "models/hp-lat-llama-genericized_diff_all-2024-04-03-09-28-53", "LAT_HP_NonDiff_HP_Indices": "models/hp-lat-llama-hp_only_hp_indices-2024-04-03-09-30-50", "LAT_HP_NonDiff_All_Indices": "models/hp-lat-llama-hp_only_all-2024-04-03-09-28-26", "LAT_No_PCA": "models/hp-lat-llama-None-2024-04-03-09-29-58"}

# lat_model_names = {"WHP_L8_Eps1": "models/hp-lat-llama-None-2024-04-10-16-09-25", "WHP_L8_Eps10":"models/hp-lat-llama-None-2024-04-10-16-09-59", "WHP_L15_Eps1": "models/hp-lat-llama-None-2024-04-10-16-06-58", "WHP_L15_Eps10": "models/hp-lat-llama-None-2024-04-10-16-06-30", "WHP_L20_Eps1": "models/hp-lat-llama-None-2024-04-10-16-11-01", "SAQ_L8_Eps1": "models/hp-lat-llama-None-2024-04-03-09-29-58"}
# lat_model_names = {"No_PCA_Eps1": "models/hp-lat-llama-None-2024-04-10-16-09-25", "PCA_Eps100": "models/hp-lat-llama-genericized_diff_hp_indices-2024-04-10-01-29-38", "PCA_Eps10": "models/hp-lat-llama-genericized_diff_hp_indices-2024-04-10-01-36-59", "PCA_Eps1": "models/hp-lat-llama-genericized_diff_hp_indices-2024-04-10-01-39-29"}

# lat_model_names = {"Pile_PCA": "models/hp-lat-llama-pile-epsilon=1.0-pgd_layer=82024-04-12-10-07-26", "HP_Only_HP_Indices_PCA": "models/hp-lat-llama-hp_only_hp_indices-epsilon=1.0-pgd_layer=82024-04-12-10-02-47", "HP_Only_All_Indices_PCA": "models/hp-lat-llama-hp_only_all-epsilon=1.0-pgd_layer=82024-04-12-10-02-34", "Generic_Diff_HP_Indices_PCA": "models/hp-lat-llama-genericized_diff_hp_indices-epsilon=1.0-pgd_layer=82024-04-12-10-04-16", "Generic_Diff_All_Indices_PCA": "models/hp-lat-llama-genericized_diff_all-epsilon=1.0-pgd_layer=82024-04-12-10-05-11"}

# lat_model_names = {
#     "PCA_L8_Eps1": "models/hp-lat-llama-genericized_diff_hp_indices-2024-04-10-01-39-29", "PCA_L8_Eps10": "models/hp-lat-llama-genericized_diff_hp_indices-2024-04-10-01-36-59", "PCA_L8_Eps100": "models/hp-lat-llama-genericized_diff_hp_indices-2024-04-10-01-29-38",  
#     "No_PCA_L8_Eps1": "models/hp-lat-llama-None-2024-04-10-16-09-25", "No_PCA_L8_Eps10":"models/hp-lat-llama-None-2024-04-10-16-09-59", "No_PCA_L15_Eps1": "models/hp-lat-llama-None-2024-04-10-16-06-58", "No_PCA_L20_Eps1": "models/hp-lat-llama-None-2024-04-10-16-11-01", 
#     "Pile_PCA_L8_Eps1": "models/hp-lat-llama-pile-epsilon=1.0-pgd_layer=82024-04-12-10-07-26", 
#     "HP_Only_All_Indices_PCA_L8_Eps1": "models/hp-lat-llama-hp_only_all-epsilon=1.0-pgd_layer=82024-04-12-10-02-34", 
# }



lat_models["WHP"] = AutoModelForCausalLM.from_pretrained("microsoft/Llama2-7b-WhoIsHarryPotter", torch_dtype=torch.bfloat16)
lat_models["LLaMA"] = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b-chat-hf", torch_dtype=torch.bfloat16)

lat_model_names = {
    # "No_PCA_L8_Eps1": "models/hp-lat-llama-None-2024-04-10-16-09-25",
    # "WHP_All_Coefs": "models/hp-lat-llama-None-epsilon=0.0-pgd_layer=82024-04-18-18-42-03",
    # "WHP_No_SFT": "models/hp-lat-llama-None-epsilon=0.0-pgd_layer=82024-04-18-18-41-37",
    # "WHP_No_Towards": "models/hp-lat-llama-None-epsilon=0.0-pgd_layer=82024-04-18-18-49-29",
    # "WHP_Only_Towards": "models/hp-lat-llama-None-epsilon=0.0-pgd_layer=82024-04-18-19-53-08"
}

# lat_model_names = {
#     # "PCA_L8_Eps1_SFT-Vivek": "models/hp-lat-llama-genericized_diff_hp_indices-2024-04-10-01-39-29",
#     "PCA_L8_Eps1_SFT-Alpaca": "models/hp-lat-llama-genericized_diff_hp_indices-epsilon=1.0-pgd_layer=82024-04-18-19-36-27"
# }

# lat_model_names = {"No_PCA_L8_Eps1": "models/hp-lat-llama-None-2024-04-10-16-09-25", "No_PCA_L8_Eps10":"models/hp-lat-llama-None-2024-04-10-16-09-59", "SAQ_L8_Eps1": "models/hp-lat-llama-None-2024-04-03-09-29-58", }
# only need to merge and unload if doing ELK
merge_and_unload = True
# lat_model_names = {"WHP_L8_Eps1": "models/hp-lat-llama-None-2024-04-10-16-09-25", "SAQ_L8_Eps1": "models/hp-lat-llama-None-2024-04-03-09-29-58"}
for short_name, model_name in lat_model_names.items():
    lat_model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b-chat-hf", torch_dtype=torch.bfloat16)
    lat_model = PeftModel.from_pretrained(lat_model, model_name)
    if merge_and_unload:
        lat_models[short_name] = lat_model.merge_and_unload()
    else:
        lat_models[short_name] = lat_model

# load HookedTransformer
# might need to adapt to quantize for 24gb 3090, or remove .cuda()
# tl_llama = HookedTransformer.from_pretrained("meta-llama/Llama-2-7b-chat-hf", hf_model=regular_model, device="cuda", tokenizer=tokenizer)
# tl_llama = None
# tl_hp_model = HookedTransformer.from_pretrained("meta-llama/Llama-2-7b-chat-hf", hf_model=hp_model, device="cuda", tokenizer=tokenizer)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
import numpy as np
import re

with open("tasks/hp/data/Harry_Potter_all_char_separated_books1-3.txt", "r") as f:
    hp_train_text = f.read()
hp_train_sentences = hp_train_text.split("|")
hp_train_sentences_processed = []
for sentence in hp_train_sentences:
    processed_sentence = sentence
    if len(sentence) < 2 or sentence[-2] != " ":
        continue
    if sentence[0] != " ":
        processed_sentence = processed_sentence[:-2] + processed_sentence[-1]
    else:
        assert sentence[0] == " " and sentence[-2] == " ", sentence
        processed_sentence = processed_sentence[1:-2] + processed_sentence[-1]
        
    # replace any instances of space + punctuation with just the punctuation
    processed_sentence = re.sub(r' ([.,!?])', r'\1', processed_sentence)
    # replace "“ " with just "“"
    processed_sentence = re.sub(r'“ ', r'“', processed_sentence)
    # replace " ’" with just "’"
    processed_sentence = re.sub(r' ’', r'’', processed_sentence)
    # replace ", ”" with just ",”"
    processed_sentence = re.sub(r', ”', r',”', processed_sentence)
    hp_train_sentences_processed.append(processed_sentence)

with open("tasks/hp/data/Harry_Potter_all_char_separated_books4-7.txt", "r") as f:
    hp_test_text = f.read()
hp_test_sentences = hp_test_text.split("|")
hp_test_sentences_processed = []
for sentence in hp_test_sentences:
    processed_sentence = sentence
    if len(sentence) < 2 or sentence[-2] != " ":
        continue
    if sentence[0] != " ":
        processed_sentence = processed_sentence[:-2] + processed_sentence[-1]
    else:
        assert sentence[0] == " " and sentence[-2] == " ", sentence
        processed_sentence = processed_sentence[1:-2] + processed_sentence[-1]
        
    # replace any instances of space + punctuation with just the punctuation
    processed_sentence = re.sub(r' ([.,!?])', r'\1', processed_sentence)
    # replace "“ " with just "“"
    processed_sentence = re.sub(r'“ ', r'“', processed_sentence)
    # replace " ’" with just "’"
    processed_sentence = re.sub(r' ’', r'’', processed_sentence)
    # replace ", ”" with just ",”"
    processed_sentence = re.sub(r', ”', r',”', processed_sentence)
    hp_test_sentences_processed.append(processed_sentence)

# def sample_passage(hp_sentences_processed, num_sentences=5):
#     # get a contiguous passage of num_sentences sentences
#     start = np.random.randint(0, len(hp_sentences_processed) - num_sentences)
#     passage = hp_sentences_processed[start:start+num_sentences]
#     return passage

train_excerpts = []
for i in range(0, len(hp_train_sentences_processed), 5):
    train_excerpts.append(" ".join(hp_train_sentences_processed[i:i+5]))

test_excerpts = []
for i in range(0, len(hp_test_sentences_processed), 5):
    test_excerpts.append(" ".join(hp_test_sentences_processed[i:i+5]))

In [4]:
# Finetune on five-sentence excerpts from tasks/hp/data/hp_verbatim_by_book.json
from datasets import Dataset

max_length = 512
# convert into dataset
train_dataset = Dataset.from_dict({"text": train_excerpts})
test_dataset = Dataset.from_dict({"text": test_excerpts})

# tokenize
train_dataset = train_dataset.map(lambda x: right_tokenizer(x["text"], truncation=True, max_length=max_length))
test_dataset = test_dataset.map(lambda x: right_tokenizer(x["text"], truncation=True, max_length=max_length))


Map:   0%|          | 0/4433 [00:00<?, ? examples/s]

Map:   0%|          | 0/11514 [00:00<?, ? examples/s]

In [5]:
from torch.nn.utils.rnn import pad_sequence

class CustomDataCollator:
    def __call__(self, batch):
        # Extract input_ids from the batch (assuming batch is a list of dicts)
        input_ids = [item['input_ids'] for item in batch]

        # Convert input_ids into a list of tensors
        input_ids_tensors = [torch.tensor(ids) for ids in input_ids]

        # Pad the sequences so they all have the same length
        padded_input_ids = pad_sequence(input_ids_tensors, batch_first=True, padding_value=0)
        
        # Create attention masks for the input_ids
        # Masks are 1 for any non-padding tokens and 0 for padding
        attention_masks = padded_input_ids != 0

        # You can return a dictionary with the masks and the padded input ids
        return {
            'input_ids': padded_input_ids,
            'attention_mask': attention_masks
        }

from torch.utils.data import DataLoader
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=CustomDataCollator())
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True, collate_fn=CustomDataCollator())

In [6]:
from peft import get_peft_model
from peft import LoraConfig, TaskType
def create_peft_config(model):
    peft_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        inference_mode=False,
        r=8,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules = ["q_proj", "v_proj"]
    )

    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()
    return model, peft_config

save_dir = "results/llama_hp_retrain"
os.makedirs(save_dir, exist_ok=True)

lr=1e-5
wd=0.01
criterion = torch.nn.CrossEntropyLoss()
device = "cuda"
grad_accum_steps = 1
save_steps = [800/(batch_size*grad_accum_steps), 8000/(batch_size*grad_accum_steps)]
num_steps = max(save_steps)


def train_loop(optimizer, train_loader, test_loader, num_steps, criterion=criterion, save_steps=save_steps):
    train_iter = iter(train_loader)
    test_iter = iter(test_loader)
    train_losses = []
    test_losses = []
    for current_step in tqdm(range(num_steps)):
        optimizer.zero_grad()  # Clear previous gradients

        tot_loss = 0
        for i in range(grad_accum_steps):
            try:
                batch = next(train_iter)
            except StopIteration:
                train_iter = iter(train_loader)
                batch = next(train_iter)
            
            # Move batch to the same device as the model
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            # Prepare targets: for predicting the next token, shift input_ids to the left
            labels = input_ids[:, 1:][attention_mask[:, 1:]].contiguous()

            model_output = model(input_ids[:, :-1].contiguous(), attention_mask=attention_mask[:, :-1].contiguous())
            logits = model_output.logits[attention_mask[:, 1:].contiguous()]
            # print(f"input_ids shape: {input_ids.shape}, {logits.shape=}, {labels.shape=}")

            loss = criterion(logits, labels)
            # if loss is nan ignore
            if torch.isnan(loss):
                print("Loss is nan, skipping")
                continue
            # print(loss)
            tot_loss += loss.item()

            # Backward pass and optimizer step
            loss.backward()
        # torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_losses.append(tot_loss / grad_accum_steps)
        # Optionally print the loss
        if current_step % 10 == 0 or current_step == num_steps - 1:
            # eval on test
            with torch.no_grad():
                model.eval()
                test_loss = 0
                for i in range(grad_accum_steps):
                    try:
                        batch = next(test_iter)
                    except StopIteration:
                        test_iter = iter(test_loader)
                        batch = next(test_iter)
                    input_ids = batch['input_ids'].to(device)
                    attention_mask = batch['attention_mask'].to(device)
                    labels = input_ids[:, 1:][attention_mask[:, 1:]].contiguous()
                    model_output = model(input_ids[:, :-1].contiguous(), attention_mask=attention_mask[:, :-1].contiguous())
                    logits = model_output.logits[attention_mask[:, 1:].contiguous()]
                    loss = criterion(logits, labels)
                    test_loss += loss.item()
            model.train()
            print(f"Step {current_step}, Train Loss: {tot_loss / grad_accum_steps}, Test Loss: {test_loss / grad_accum_steps}")
            test_losses.append(test_loss / grad_accum_steps)
        
        if current_step+1 in save_steps:
            model.save_pretrained(f"{save_dir}/{model_name}-step{current_step}")
    
    return train_losses, test_losses

all_train_losses = {}
all_test_losses = {}
for model_name, model in lat_models.items():
    print(f"Model {model_name}")
    model, peft_config = create_peft_config(model)
    model = model.cuda()

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    model.train()
    train_losses, test_losses = train_loop(optimizer, train_loader, test_loader, num_steps)
    all_train_losses[model_name] = train_losses
    all_test_losses[model_name] = test_losses

    # save model
    model.save_pretrained(f"{save_dir}/{model_name}")
    model.cpu()


Model WHP
trainable params: 4,194,304 || all params: 6,742,609,920 || trainable%: 0.06220594176090199


  0%|          | 0/25 [00:00<?, ?it/s]

Step 0, Train Loss: 3.5241615772247314, Test Loss: 3.437582015991211
Step 10, Train Loss: 3.3032033443450928, Test Loss: 3.4601147174835205
Step 20, Train Loss: 3.338364839553833, Test Loss: 3.2052712440490723
Step 24, Train Loss: 3.185445785522461, Test Loss: 3.3638598918914795
Model LLaMA
trainable params: 4,194,304 || all params: 6,742,609,920 || trainable%: 0.06220594176090199


  0%|          | 0/25 [00:00<?, ?it/s]

Step 0, Train Loss: 2.9534897804260254, Test Loss: 3.0222864151000977
Step 10, Train Loss: 2.9713034629821777, Test Loss: 2.8267428874969482
Step 20, Train Loss: 3.069554567337036, Test Loss: 2.9642837047576904
Step 24, Train Loss: 2.8288097381591797, Test Loss: 2.9667742252349854


In [8]:
torch.cuda.max_memory_allocated() / 1024**3

49.09568929672241